# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

from promptpotter.application.campaign.config import (
    configure_and_apply_pipeline,
    load_campaign_config,
)
from promptpotter.application.campaign.data import prepare_datasets
from promptpotter.application.campaign.utils import save_campaign_winner
from promptpotter.application.optimization.pipeline import decompose_task_context
from promptpotter.presentation.views.display_primitives import set_display_tags
from promptpotter.presentation.views.reports import (
    render_campaign_summary,
    render_experiment_dashboard,
    render_flip_tracking,
    render_lineage,
    render_preflight,
)

from notebooks._dev_reload import dev_reload
from promptpotter.infrastructure.tracing import sync_langfuse_runs
from promptpotter.presentation.views.notebook_run import (
    init_notebook_session,
    prepare_origin_notebook,
    run_optimization_notebook,
)

# --- Services ---
session = await init_notebook_session()
TASK_DESCRIPTION = Path(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
).read_text(encoding="utf-8")

# --- Campaign config ---
# Tunable knobs live here. Loop invariants (enable_l2/l3, scoring-set thresholds, etc.)
# are defaults in OptimizationConfig and do not belong here.
campaign_config = load_campaign_config(
    {
        "exclude_nodes": ["llm_ranking"],
        # --- Backend node overrides ---
        # Override values from GET /pipeline. Nested format: {"node": {"param": value}}.
        "pipeline_overrides": {
            "web_search": {"max_sites": 20, "num_results": 20, "content_char_limit": 800},
            "entity_profiling": {"model": "openai/gpt-oss-120b", "max_tokens": 4000},
            "llm_ranking": {"model": "openai/gpt-oss-120b"},
        },
        "sp_budget_ttest": 15,
        "optimization": {
            # --- Core loop ---
            "l1_patience": 2,
            "max_rounds": None,
            "n_variants": 5,
            "creativity": 0.7,
            "improvement_threshold": 0.01,
            # --- Escalation ---
            "degradation_threshold": 0.4,
            "l2_patience": 2,
            "l3_patience": 1,
            "l2_temperature": 0.3,
            "l3_temperature": 0.5,
        },
        "optimizer_llm": {
            "model": "openai/gpt-oss-120b",
            "provider": "groq",
            "temperature": 0.4,
            "max_tokens": 2000,
        },
    }
)

# --- Pipeline snapshot & params ---
pipeline_config_full = await session.backend_client.fetch_pipeline()
print(json.dumps(pipeline_config_full.get("data", pipeline_config_full), indent=2))
pipeline_params = configure_and_apply_pipeline(session, campaign_config, log=print)
set_display_tags(session.pipeline_schema)

In [ ]:
# @title Load data & evaluation context

result = prepare_datasets(
    session.store,
    excel_path=r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx",
)
train_data = result.train_data or []
session.index_terms = result.index_terms
print(
    f"Train: {len(result.splits.get('train', []))}  "
    f"Test (processes): {len(result.splits.get('test_processes', []))}  "
    f"Test (material): {len(result.splits.get('test_material', []))}"
)
print(f"Combined samples: {result.n_unique_samples}  Session terms: {len(result.index_terms)}")

origin_ps, dataset, campaign_rounds, origin_results = await prepare_origin_notebook(
    session,
    train_data,
    campaign_config,
    pipeline_params=pipeline_params,
)

In [ ]:
# @title Experiment dashboard
EXPERIMENT_ID = None  # Set to hex ID to resume (e.g. '68e2c5')
pipeline_params = locals().get("pipeline_params")  # preserve on re-run

text, pipeline_params, campaign_config = render_experiment_dashboard(
    session=session,
    experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config,
    dataset=dataset,
    origin_prompt_fields=campaign_rounds[0]["prompt_fields"].model_dump()
    if campaign_rounds
    else None,
    pipeline_params=pipeline_params,
)
print(text)

## 3. Explore

Exploration via **Smart Search** (scan advisor + sensitivity scan).

In [ ]:
# @title Task context

from promptpotter.application.campaign.config import create_llm_client

_llm_client, _llm_model = create_llm_client(campaign_config)
task_context, _consultation, _was_cached = await decompose_task_context(
    TASK_DESCRIPTION,
    _llm_client,
    _llm_model,
    store_base_dir=session.store.base_dir,
    backend_id=session.backend_id,
)
for f in task_context.FIELDS:
    val = getattr(task_context, f, "")
    print(f"  {f}: {val or '(empty)'}")

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
# @title Feedback cycle preflight
print(
    render_preflight(
        campaign_config,
        session,
        campaign_rounds,
        dataset,
        pipeline_schema=session.pipeline_schema,
    )
)

In [ ]:
# @title Run optimization (feedback cycle)
dev_reload()

campaign_rounds, _cycle_result = await run_optimization_notebook(
    campaign_rounds,
    dataset,
    campaign_config,
    session=session,
    experiment_id=EXPERIMENT_ID,
    task_context=task_context,
)

In [ ]:
# @title 5. Results -- summary, save, sync
print(render_campaign_summary(campaign_rounds))
print(
    render_flip_tracking(campaign_rounds)
    or "Need at least 2 rounds with results for flip tracking."
)
print(render_lineage(campaign_rounds))

# --- Persist (T2: below the fold) ---
save_campaign_winner(
    campaign_rounds,
    campaign_config,
    session.store,
    session.backend_id,
    campaign_id=EXPERIMENT_ID,
)
sync_result = sync_langfuse_runs(
    session.store,
    session.backend_id,
    dataset_name="termnorm_ground_truth",
)
if sync_result is None:
    print("No completed dataset runs yet -- skipping Langfuse backfill.")